# 07 External tests
(a) HCP 7T diffusion of the test subjects: another scanner, resolution and set of b-values. (b) HCP test-retest. (c) NSD: other people, another site and protocol, both diffusion runs.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy
from retinopath.train import load_trained, net_fodf_all, trained_runs
subjects = PL.make_subject_lists(cfg); runs = [run for run in trained_runs(cfg) if run[0] in ("path", "voxel", "signal", "mlp")]
sessions_7t = [PL.session(sid, "7T") for sid in subjects["t7"]]
with Timer(cfg, "07_hcp7t"):
    for variant, seed in runs:
        variant_cfg = copy.deepcopy(cfg); model, _ = load_trained(variant_cfg, variant, seed, device)
        net_fodf_all(variant_cfg, sessions_7t, ["NATIVE"], f"net_{variant}_s{seed}", model, device)
    scores_7t = PL.run_grid(cfg, sessions_7t, [("NATIVE", "csd")] + [("NATIVE", f"net_{variant}_s{seed}") for variant, seed in runs], "external_7t", ref=None)
scores_7t[["sid", "method", "n_pairs", "n_valid", "rsi_ree_all", "atlas_rsi_ree_all"]].round(3)

In [ ]:
# Test-retest of the reference pipeline (second visit, own anatomy, same vertices)
sessions_retest = [PL.session(sid, "retest") for sid in subjects["retest"]]
with Timer(cfg, "07_retest"):
    scores_retest = PL.run_grid(cfg, sessions_retest, [("REF", "msmt_csd")], "retest_rsi", ref=None)
    retest_overlap = pd.DataFrame(PL.each(cfg, PL.retest_scores, subjects["retest"], "retest overlap")); retest_overlap.to_csv(RES / "retest_overlap.csv", index=False)
retest_overlap.round(3)

In [ ]:
# NSD: 8 other people, another scanner and protocol (1.5 mm, b = 1500 and 3000), their own measured pRF maps and
# hand-drawn V1-V3. Nothing here was simulated or seen in training. Downloads about 4 GB per subject if missing.
sessions_nsd = [PL.session(sid, "nsd") for sid in subjects.get("nsd", [])]
with Timer(cfg, "07_nsd"):
    PL.prepare_all(cfg, sessions_nsd, "NSD crops")
    for variant, seed in runs:
        variant_cfg = copy.deepcopy(cfg); model, _ = load_trained(variant_cfg, variant, seed, device)
        net_fodf_all(variant_cfg, sessions_nsd, ["NATIVE"], f"net_{variant}_s{seed}", model, device)
    scores_nsd = PL.run_grid(cfg, sessions_nsd, [("NATIVE", "csd"), ("NATIVE", "csd+prob"), ("NATIVE", "proximity")] + [("NATIVE", f"net_{variant}_s{seed}") for variant, seed in runs], "external_nsd", ref=None) if sessions_nsd else None
scores_nsd[["sid", "method", "n_pairs", "n_valid", "ree_median_all", "rsi_ree_all", "rsi_ree_geodesic_all", "template_rsi_ree_all", "atlas_rsi_ree_all"]].round(3) if sessions_nsd else print("NSD: NOT RUN (n_nsd is 0)")

In [ ]:
# NSD run-to-run reproducibility: each subject has a second diffusion run in the same space
if sessions_nsd and cfg.nsd.use_run2:
    sessions_nsd2 = [PL.session(sid, "nsd2") for sid in subjects["nsd"]]
    with Timer(cfg, "07_nsd_run2"):
        PL.prepare_all(cfg, sessions_nsd2, "NSD run 2 crops")
        scores_nsd2 = PL.run_grid(cfg, sessions_nsd2, [("NATIVE", "csd")], "external_nsd_run2", ref=None)
        nsd_overlap = pd.DataFrame(PL.each(cfg, lambda worker_cfg, sid: PL.retest_scores(worker_cfg, sid, "NATIVE", "csd", kinds=("nsd", "nsd2")), subjects["nsd"], "NSD run overlap"))
    nsd_overlap["rsi_run1"] = scores_nsd[scores_nsd.method == "csd"].set_index("sid").rsi_ree_all.reindex(nsd_overlap.sid).to_numpy(); nsd_overlap["rsi_run2"] = scores_nsd2.set_index("sid").rsi_ree_all.reindex(nsd_overlap.sid).to_numpy()
    nsd_overlap.to_csv(RES / "nsd_run_overlap.csv", index=False); display(nsd_overlap.round(3))
else:
    print("NSD second run: NOT RUN")